# Experiment 6 · Cine-clip video propagation: frame-by-frame vs video

This experiment is unique to the benchmark: it evaluates the promptable memory-based
video segmenters — **SAM2**, **MedSAM-2**, and **SAM3** — on the **Stanford AIMI thyroid
cine-clips**, contrasting two inference regimes. In **framewise** mode each frame gets its
own independent box prompt (the same protocol as the static experiments). In **video** mode
only the *first* frame is box-prompted; the mask is then propagated through the clip via the
model's memory bank (`propagate_in_video`). Comparing the two isolates the effect of temporal
propagation on both spatial accuracy (DSC / IoU / HD95) and **temporal consistency** — the
inter-frame Jaccard \(J_t\) between consecutive predicted masks. Three models times two
modes gives the 2x2-plus matrix behind Table 5 and Figure 5. All runs use a fixed **seed 42**,
512x512 frames, and a 10% oracle-box jitter aligned frame-for-frame between the two modes so
the comparison is *same prompt, different inference*.

The experiment is run at **three supervision levels**: the native zero-shot checkpoint, and the LoRA adapters at `f=5%` and `f=100%` (`--fraction 0.05` / `--fraction 1.0`, folded in from Experiments 3 and 1 respectively). That ladder is what separates *the propagation mechanism is weak* from *the model is simply under-adapted to this data*.


In [ ]:
# Move to the repository root (the directory that contains pyproject.toml) so that
# the `thyroidbench` package is importable and the relative --data_root / --split_dir /
# --video_dir defaults resolve correctly.
import os
from pathlib import Path

cwd = Path.cwd().resolve()
for candidate in [cwd, *cwd.parents]:
    if (candidate / "pyproject.toml").exists():
        os.chdir(candidate)
        break
print("Repository root:", Path.cwd())

In [ ]:
# Check the inputs this experiment needs before doing anything slow. A missing
# dataset here means an unrun (or unplaced) setup step, not a bug in the experiment.
from pathlib import Path

REQUIRED = ['ddti', 'tn3k', 'thyroidxl', 'stanford_aimi']
GATED = {'thyroidxl', 'stanford_aimi'}

missing = [d for d in REQUIRED
           if not (Path('data/processed') / d / 'images').is_dir()
           or not (Path('data/splits') / f'{d}_test.csv').exists()]
if missing:
    print('Missing preprocessed data or splits for:', ', '.join(missing))
    for d in missing:
        if d in GATED:
            print(f'  {d:14s} access-gated -> place your approved copy first, see '
                  f'00_setup/01_place_gated_datasets.ipynb')
        else:
            print(f'  {d:14s} open -> download it with 00_setup/00_get_open_datasets.ipynb')
    print('Then run 00_setup/02_preprocess.ipynb and 00_setup/03_make_splits.ipynb.')
    print('\nYou can still run this experiment on whichever datasets ARE present '
          'by restricting the --dataset argument below.')
else:
    print('All four datasets are preprocessed and split.')


## Prerequisites

- **Stanford AIMI cine-clips** must be downloaded and preprocessed into `data/processed/stanford_aimi/`
  (images + `masks/`), with the test split listed in `data/splits/stanford_aimi_test.csv`. This dataset
  is access-gated; see the top-level data README for the request procedure.
- **Video frames** for the propagation mode must be built with `prepare_videos.py` (next cell). It writes
  one per-patient directory of zero-indexed JPEGs (`00000.jpg`, `00001.jpg`, ...) plus a
  `frame_index_map.json` sidecar under `data/cineclip_videos/`, matching the numeric-filename convention
  SAM2's video loader requires.
- **Foundation-model weights** are not shipped with the repository. Download the published checkpoints
  and place them under a git-ignored `pretrained_models/` directory at the repository root
  (`pretrained_models/sam2/sam2.1_hiera_large.pt`, `pretrained_models/medsam2/MedSAM2_latest.pt`, and the
  SAM3 weights resolved by `thyroidbench/models/sam3_video_wrapper.py`).

In [ ]:
# !python experiments/exp6_cineclip_video/prepare_videos.py

## Run

Three supervision levels x three models x two modes = 18 runs. Each call writes
`results/<model>_<mode>[_f<fraction>]/per_frame_metrics.csv`, `per_sequence_metrics.csv`
and `summary.json`. The zero-shot pass needs no adapter; the two LoRA passes load the
adapter checkpoints trained by Experiment 3 (`f=0.05`) and Experiment 1 (`f=1.0`), so run
those first if the checkpoints are not on disk.


In [ ]:
# Zero-shot: the native checkpoint, no adapter.
for model in ['sam2', 'medsam2', 'sam3']:
    for mode in ['framewise', 'video']:
        !python experiments/exp6_cineclip_video/run.py --model {model} --mode {mode}

# LoRA-adapted: f=5% and f=100%.
for fraction in [0.05, 1.0]:
    for model in ['sam2', 'medsam2', 'sam3']:
        for mode in ['framewise', 'video']:
            !python experiments/exp6_cineclip_video/run.py --model {model} --mode {mode} --fraction {fraction}


## Aggregate

Three steps, in order:

1. `aggregate_zeroshot.py` folds the six zero-shot cells into `zeroshot_summary.csv`,
   `zeroshot_paired_per_patient.csv` and `zeroshot_collapse_summary.csv`.
2. `aggregate_adapted.py` does the same for the twelve LoRA cells (`adapted_*.csv`).
3. `build_ladder.py` reconciles all three tiers into `ladder_summary.csv` and runs the
   paired f=5% vs f=100% Wilcoxon test on per-patient video DSC.

`stats_zeroshot.py` additionally computes bootstrap 95% CIs and the paired
framewise-vs-video Wilcoxon tests for the zero-shot tier.

The per-patient intermediates are keyed by Stanford AIMI patient id, so they stay local:
only the aggregate tables are committed.


In [ ]:
!python experiments/exp6_cineclip_video/aggregate_zeroshot.py
!python experiments/exp6_cineclip_video/aggregate_adapted.py
!python experiments/exp6_cineclip_video/build_ladder.py


## Results

`ladder_summary.csv` is the headline table: mean framewise and video DSC per
(model, supervision tier), with the patient-level collapse rate (share of clips losing
at least 0.10 DSC when the per-frame prompts are replaced by a single frame-0 prompt).
`zeroshot_summary.csv` and `adapted_summary.csv` carry the fuller per-mode metrics
(IoU, HD95, inter-frame consistency, empty-prediction rate).


In [ ]:
import pandas as pd

ladder = pd.read_csv('experiments/exp6_cineclip_video/results/ladder_summary.csv')
view = ladder[['model', 'tier', 'n', 'dsc_framewise_mean', 'dsc_video_mean',
               'delta_dsc_mean', 'collapse_rate_pct']].round(3)
display(view)

wilcoxon = pd.read_csv('experiments/exp6_cineclip_video/results/ladder_5v100_wilcoxon.csv')
display(wilcoxon.round(4))
